In [4]:
# ==========================================================
# YOLOv11-S TRAINING (PRECISION OPTIMIZED VERSION)
# Cross Validation Ready (M1–M5)
# NO GOOGLE DRIVE REQUIRED
# ==========================================================

# =========================
# 1️⃣ Install Dependencies
# =========================
!pip install -U ultralytics pyyaml -q

import os
import zipfile
import yaml
import torch
from ultralytics import YOLO
from google.colab import files

print("✅ Ultralytics Installed")
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# =========================
# 2️⃣ Upload ZIP Dataset
# =========================
print("\n⬆️ Upload your dataset ZIP (M1 or M2 or M3...)")
uploaded = files.upload()

zip_filename = list(uploaded.keys())[0]
print("Uploaded:", zip_filename)


# =========================
# 3️⃣ Extract Dataset
# =========================
dataset_path = "/content/dataset"

if os.path.exists(dataset_path):
    !rm -rf /content/dataset

os.makedirs(dataset_path, exist_ok=True)

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(dataset_path)

print("✅ Dataset extracted")


# =========================
# 4️⃣ Locate data.yaml
# =========================
yaml_file = None

for root, dirs, files_ in os.walk(dataset_path):
    if "data.yaml" in files_:
        yaml_file = os.path.join(root, "data.yaml")
        dataset_root = root
        break

if yaml_file is None:
    raise Exception("❌ data.yaml not found!")

print("Found data.yaml:", yaml_file)


# =========================
# 5️⃣ Fix Paths Automatically
# =========================
with open(yaml_file, "r") as f:
    data_config = yaml.safe_load(f)

data_config["path"] = dataset_root
data_config["train"] = "train/images"
data_config["val"] = "valid/images"
data_config["test"] = "test/images"

with open(yaml_file, "w") as f:
    yaml.dump(data_config, f)

print("✅ data.yaml paths fixed")


# =========================
# 6️⃣ Load YOLOv11-s
# =========================
model = YOLO("yolo11s.pt")


# =========================
# 7️⃣ TRAIN MODEL (PRECISION OPTIMIZED)
# =========================
results = model.train(

    data=yaml_file,

    # -------- Training --------
    epochs=300,
    imgsz=640,
    batch=4,

    # -------- Optimizer --------
    optimizer="AdamW",
    lr0=0.0008,
    lrf=0.01,
    weight_decay=0.0005,

    # -------- Transfer Learning --------
    freeze=15,

    # -------- Early stopping --------
    patience=80,

    # -------- LIGHT AUGMENTATION (KEY FIX) --------
    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=2,
    translate=0.05,
    scale=0.3,
    shear=1,
    perspective=0.0003,
    fliplr=0.5,

    mosaic=0.5,     # reduced
    mixup=0.0,      # disabled (BIG for precision)

    # -------- Stability --------
    cos_lr=True,
    warmup_epochs=5,

    # -------- Saving --------
    project="/content/runs",
    name="precision_improved_run",
    exist_ok=True,
    device=0 if torch.cuda.is_available() else "cpu",
    plots=True,
    verbose=True
)

print("✅ Training completed")


# =========================
# 8️⃣ VALIDATION RESULTS
# =========================
metrics = model.val()

print("\n===== VALIDATION METRICS =====")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"mAP@0.5: {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")


# =========================
# 9️⃣ TEST RESULTS
# =========================
test_metrics = model.val(data=yaml_file, split="test")

print("\n===== TEST METRICS =====")
print(f"Precision: {test_metrics.box.mp:.4f}")
print(f"Recall: {test_metrics.box.mr:.4f}")
print(f"mAP@0.5: {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {test_metrics.box.map:.4f}")


# =========================
# 🔟 Download best.pt
# =========================
best_model_path = "/content/runs/precision_improved_run/weights/best.pt"

print("\n⬇️ Downloading best.pt ...")
files.download(best_model_path)

✅ Ultralytics Installed
GPU Available: True
GPU: Tesla T4

⬆️ Upload your dataset ZIP (M1 or M2 or M3...)


Saving M1_Charoenphool Intersection.v1i.yolov11.zip to M1_Charoenphool Intersection.v1i.yolov11 (1).zip
Uploaded: M1_Charoenphool Intersection.v1i.yolov11 (1).zip
✅ Dataset extracted
Found data.yaml: /content/dataset/data.yaml
✅ data.yaml paths fixed
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/data.yaml, degrees=2, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=15, half=False, hsv_h=0.01, hsv_s=0.5, hsv_v=0.3, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>